# The Wild Project — Jordi Wild Avatar (callLQ)

Voice chatbot powered by:
- **faster-whisper** — speech-to-text (Spanish)
- **Claude claude-sonnet-4-6** — response generation grounded in real Jordi quotes
- **edge-tts** — online Spanish TTS (`es-ES-AlvaroNeural`)
- **TF-IDF RAG** — retrieves the most relevant real Jordi lines before each Claude call

Runtime: CPU only, no GPU required.

---
**Before running:** add your `ANTHROPIC_API_KEY` in Colab Secrets (key icon in the left sidebar).

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install -q anthropic faster-whisper edge-tts scikit-learn numpy python-dotenv gradio

In [ ]:
# ── 2. Clone repo & rebuild knowledge base ───────────────────────────────────
import os

if not os.path.exists("wild-project-avatar"):
    !git clone -q https://github.com/fborbon/wild-project-avatar.git

os.chdir("wild-project-avatar")
print("Repo ready:", os.getcwd())

# Rebuild the TF-IDF index from committed jordi_lines.txt (takes ~5s, no API calls)
!python3 06_build_knowledge_base.py --skip-extract

In [ ]:
# ── 3. API key ────────────────────────────────────────────────────────────────
import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("API key loaded from Colab Secrets.")
except Exception:
    import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")

In [ ]:
# ── 4. Load models & data ─────────────────────────────────────────────────────
import pickle, json, re, tempfile
import numpy as np
import anthropic
from faster_whisper import WhisperModel
from sklearn.metrics.pairwise import cosine_similarity

print("Loading Whisper base (int8, CPU)...")
whisper_model = WhisperModel("base", device="cpu", compute_type="int8")
print("  Done.")

print("Loading knowledge base...")
with open("data/knowledge_base.pkl", "rb") as f:
    kb = pickle.load(f)
print(f"  {len(kb['entries'])} entries loaded.")

with open("data/jordi_style_profile.json", encoding="utf-8") as f:
    profile = json.load(f)

facts = {}
if os.path.exists("data/jordi_facts.json"):
    with open("data/jordi_facts.json", encoding="utf-8") as f:
        facts = json.load(f)

client = anthropic.Anthropic()
print("All models ready.")

In [ ]:
# ── 5. Core logic (RAG + prompts) ─────────────────────────────────────────────

CONTENT_SAFETY = """
POLÍTICA DE CONTENIDO (obligatoria, sin excepciones):
- Puedes dar opiniones directas y sinceras, incluso polémicas, como lo haría Jordi.
- NUNCA ataques a personas por su origen étnico, religión, género, orientación sexual u otras características personales.
- En temas sensibles (religión, política, migración, género): da tu perspectiva honesta con respeto hacia las personas.
- Puedes criticar ideas, comportamientos o instituciones, pero no a colectivos de personas.
- Esta aplicación es pública y de código abierto.
"""

SYSTEM_BASE = """Eres Jordi Wild, presentador de The Wild Project, uno de los podcasts más populares en español.

PERSONALIDAD Y ESTILO:
{style_summary}

{content_safety}

INSTRUCCIONES DE RESPUESTA:
- Responde SIEMPRE basándote en lo que realmente has dicho o piensas según los ejemplos reales.
- Si no tienes información real sobre algo, dilo honestamente: "No he hablado mucho de eso, pero..."
- Respuestas directas, sin rodeos, en español coloquial.
- Sin asteriscos, sin listas con bullets, sin markdown. Texto natural como en una conversación.

Tema de la conversación: {topic}"""

RAG_CONTEXT = """
COSAS QUE HAS DICHO REALMENTE SOBRE ESTE TEMA (úsalas como referencia):
{quotes}

Basándote en estos ejemplos reales, responde de forma consistente con lo que realmente piensas y dices.
"""


def build_style_summary(p: dict) -> str:
    if not p:
        return "Directo, curioso, irreverente, con humor. Español coloquial."
    vocab     = ", ".join(p.get("vocabulary", [])[:20])
    reactions = ", ".join(p.get("reactions", [])[:12])
    traits    = ", ".join(p.get("personality_traits", []))
    tone      = p.get("tone_description", "")
    humor     = p.get("humor_style", "")
    return f"Tono: {tone}\nRasgos: {traits}\nHumor: {humor}\nVocabulario: {vocab}\nReacciones: {reactions}"


def format_facts(f: dict) -> str:
    if not f:
        return ""
    bio   = f.get("biography", {})
    pod   = f.get("podcast", {})
    pos   = f.get("posturas_conocidas", {})
    no_es = f.get("no_es", [])
    lines = ["HECHOS VERIFICADOS SOBRE TI (máxima prioridad — nunca los contradigas):"]
    lines.append("Biografía:")
    for k, v in bio.items():
        lines.append(f"  - {k}: {v}")
    lines.append("Podcast:")
    for k, v in pod.items():
        lines.append(f"  - {k}: {v}")
    lines.append("Posturas conocidas:")
    for k, v in pos.items():
        lines.append(f"  - {k}: {v}")
    if no_es:
        lines.append("IMPORTANTE — lo que NO eres:")
        for item in no_es:
            lines.append(f"  - {item}")
    lines.append("\nSi te preguntan sobre tu vida o carrera, usa SOLO estos hechos.")
    return "\n".join(lines)


def build_system(topic: str) -> str:
    style  = build_style_summary(profile)
    system = SYSTEM_BASE.format(style_summary=style, content_safety=CONTENT_SAFETY, topic=topic)
    facts_block = format_facts(facts)
    if facts_block:
        system = facts_block + "\n\n" + system
    return system


def retrieve(query: str, n: int = 6) -> list:
    q_vec = kb["vectorizer"].transform([query])
    sims  = cosine_similarity(q_vec, kb["matrix"]).flatten()
    idxs  = sims.argsort()[-n:][::-1]
    return [{**kb["entries"][i], "score": float(sims[i])} for i in idxs if sims[i] > 0.05]


print("Core logic ready.")

In [ ]:
# ── 6. Gradio interface ────────────────────────────────────────────────────────
import gradio as gr
import edge_tts
import asyncio
import tempfile, os


async def synthesize(text: str) -> str:
    tmp = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False)
    tmp.close()
    await edge_tts.Communicate(text, voice="es-ES-AlvaroNeural").save(tmp.name)
    return tmp.name


def transcribe(audio_path: str) -> str:
    segments, _ = whisper_model.transcribe(audio_path, language="es", vad_filter=True)
    return " ".join(s.text.strip() for s in segments).strip()


async def respond(audio_path, api_history, topic):
    """Main turn handler: audio → text → RAG → Claude → TTS → audio."""
    if audio_path is None:
        return api_history, gr.update(), None, ""

    # STT
    user_text = transcribe(audio_path)
    if not user_text:
        return api_history, gr.update(), None, "⚠️ No se detectó voz. Intenta de nuevo."

    # RAG retrieval
    relevant = retrieve(user_text)
    rag_context = ""
    if relevant:
        quotes = "\n".join(f'- "{r["text"]}"' for r in relevant)
        rag_context = RAG_CONTEXT.format(quotes=quotes)

    # Claude
    messages = api_history + [{"role": "user", "content": user_text}]
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=600,
        system=build_system(topic) + rag_context,
        messages=messages,
    )
    jordi_text = response.content[0].text

    # Update history
    new_history = api_history + [
        {"role": "user",      "content": user_text},
        {"role": "assistant", "content": jordi_text},
    ]

    # TTS
    audio_out = await synthesize(jordi_text)

    # Build chatbot display [(user, bot), ...]
    display = []
    for i in range(0, len(new_history) - 1, 2):
        u = new_history[i]["content"]
        b = new_history[i + 1]["content"]
        display.append((u, b))

    status = f'🎤 Tú: "{user_text}"'
    return new_history, display, audio_out, status


async def start_conversation(topic, api_history):
    """Generate Jordi's opening line."""
    relevant = retrieve(f"bienvenida presentación podcast {topic}", n=3)
    rag_context = ""
    if relevant:
        quotes = "\n".join(f'- "{r["text"]}"' for r in relevant)
        rag_context = RAG_CONTEXT.format(quotes=quotes)

    messages = [{"role": "user", "content": "[El invitado entra al estudio.]"}]
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=600,
        system=build_system(topic) + rag_context,
        messages=messages,
    )
    jordi_text = response.content[0].text

    new_history = [
        {"role": "user",      "content": "[El invitado entra al estudio.]"},
        {"role": "assistant", "content": jordi_text},
    ]
    audio_out = await synthesize(jordi_text)
    display   = [(None, jordi_text)]
    return new_history, display, audio_out, ""


# ── UI layout ──────────────────────────────────────────────────────────────────
with gr.Blocks(title="The Wild Project — Jordi Wild Avatar", theme=gr.themes.Soft()) as demo:

    gr.Markdown("# 🎙️ The Wild Project — Jordi Wild Avatar (callLQ)")
    gr.Markdown("Habla por el micrófono y Jordi te responde con su voz.")

    api_history = gr.State([])

    with gr.Row():
        topic_input = gr.Textbox(
            label="Tema de la conversación",
            value="lo que quieras hablar",
            scale=4,
        )
        start_btn = gr.Button("▶ Iniciar conversación", variant="primary", scale=1)

    chatbot = gr.Chatbot(label="Conversación", height=380, bubble_full_width=False)

    with gr.Row():
        audio_in  = gr.Audio(sources=["microphone"], type="filepath", label="Tu voz")
        audio_out = gr.Audio(label="Jordi responde", autoplay=True)

    status_box = gr.Textbox(label="Transcripción", interactive=False, lines=1)

    with gr.Row():
        send_btn  = gr.Button("Enviar", variant="primary")
        clear_btn = gr.Button("Nueva conversación")

    # ── Event handlers ─────────────────────────────────────────────────────────
    start_btn.click(
        fn=start_conversation,
        inputs=[topic_input, api_history],
        outputs=[api_history, chatbot, audio_out, status_box],
    )

    send_btn.click(
        fn=respond,
        inputs=[audio_in, api_history, topic_input],
        outputs=[api_history, chatbot, audio_out, status_box],
    )

    # Also send when recording stops
    audio_in.stop_recording(
        fn=respond,
        inputs=[audio_in, api_history, topic_input],
        outputs=[api_history, chatbot, audio_out, status_box],
    )

    clear_btn.click(
        fn=lambda: ([], [], None, ""),
        outputs=[api_history, chatbot, audio_out, status_box],
    )

demo.launch(share=True)